# 2D multivariate random fields

Run the six sampling methods using the single shared `PCA_Theta.py` module. The model and all experiment settings are defined below. `reference` mode loads existing results; `smoke` runs a short validation; `full` runs the paper experiment.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, sys, json
import numpy as np
HERE = Path.cwd()
if not (HERE.parent / 'PCA_Theta.py').exists():
    HERE = HERE / 'RandomField_2D'
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
from PCA_Theta import METHODS, run_2d, save_result, load_result

## 1. Model

In [2]:
import numpy as np

EXAMPLE_NAME = "random_field"
BOUNDS = [(0.0, 1.0), (0.0, 1.0)]


def make_target_corr(n_outputs):
    group_ends = [0, min(8, n_outputs), min(12, n_outputs), min(16, n_outputs), min(20, n_outputs)]
    within = [0.95, 0.95, 0.85, 0.85]
    between = np.array([
        [1.00, 0.1, 0.05, 0.02],
        [0.1, 1.00, 0.08, 0.03],
        [0.05, 0.08, 1.00, 0.05],
        [0.02, 0.03, 0.05, 1.00],
    ])
    C = np.eye(n_outputs)
    groups = [range(group_ends[g], group_ends[g + 1]) for g in range(4)]
    for g, indices in enumerate(groups):
        for i in indices:
            for j in indices:
                if i != j:
                    C[i, j] = within[g]
    for g in range(4):
        for h in range(g + 1, 4):
            for i in groups[g]:
                for j in groups[h]:
                    C[i, j] = C[j, i] = between[g, h]
    return C


def matrix_square_root(C):
    values, vectors = np.linalg.eigh(C)
    return vectors @ np.diag(np.sqrt(np.maximum(values, 1e-12))) @ vectors.T


def modes(modes_per_dim):
    return [
        (kind, m, n)
        for m in range(1, modes_per_dim + 1)
        for n in range(1, modes_per_dim + 1)
        for kind in ("ss", "cs", "sc")
    ]


def evaluate_modes(X, mode_list):
    X = np.asarray(X, dtype=float)
    x1, x2 = X[:, 0], X[:, 1]
    Phi = np.empty((len(X), len(mode_list)))
    for k, (kind, m, n) in enumerate(mode_list):
        if kind == "ss":
            Phi[:, k] = np.sin(m * np.pi * x1) * np.sin(n * np.pi * x2)
        elif kind == "cs":
            Phi[:, k] = np.cos(m * np.pi * x1) * np.sin(n * np.pi * x2)
        else:
            Phi[:, k] = np.sin(m * np.pi * x1) * np.cos(n * np.pi * x2)
    return Phi


def build_model(args):
    rng = np.random.default_rng(args.field_seed)
    mode_list = modes(args.modes_per_dim)
    weights = np.asarray([
        np.exp(-args.corr_decay * (m**2 + n**2))
        for _, m, n in mode_list
    ])
    weights /= weights.sum()
    C_target = make_target_corr(args.n_outputs)
    A = rng.standard_normal((len(mode_list), args.n_outputs)) @ matrix_square_root(C_target).T
    coefficient_matrix = np.sqrt(weights)[:, None] * A

    # Match the earlier implementation: standardize each output using a fixed dense grid.
    axis = np.linspace(0.0, 1.0, 80)
    X1, X2 = np.meshgrid(axis, axis)
    calibration = np.column_stack([X1.ravel(), X2.ravel()])
    raw = evaluate_modes(calibration, mode_list) @ coefficient_matrix
    mean = raw.mean(axis=0)
    std = np.maximum(raw.std(axis=0, ddof=1), np.finfo(float).tiny)

    def evaluate(X):
        values = evaluate_modes(X, mode_list) @ coefficient_matrix
        return (values - mean) / std

    metadata = {
        "field_seed": args.field_seed,
        "n_outputs": args.n_outputs,
        "modes_per_dim": args.modes_per_dim,
        "corr_decay": args.corr_decay,
        "modes": mode_list,
        "weights": weights,
        "coefficient_matrix": coefficient_matrix,
        "target_correlation": C_target,
        "field_mean": mean,
        "field_std": std,
    }
    return evaluate, metadata

## 2. Settings

In [3]:
# reference: inspect saved paper results; smoke: N=30 to 40, one repeat;
# full: use the paper settings below. Long runs begin only in full mode.
RUN_MODE = 'reference'
REPEATS = 50
VARIANTS = [('loo', 'common')]
METHODS_TO_RUN = list(METHODS)
SETTINGS = dict(n_init=30, n_final=1000, batch_size=10,
                p_min=5, p_max=25, n_grid=100, n_test=1000,
                test_seed=2026, pca_threshold=0.01)
OUTPUT = Path(os.environ.get('PCA_THETA_OUTPUT', HERE/'Results'))

## 3. Run and save

In [4]:
if RUN_MODE not in ('reference', 'smoke', 'full'):
    raise ValueError('Choose reference, smoke, or full.')
if RUN_MODE == 'reference':
    print('Reference results are in Data/. Open Plotting.ipynb to reproduce the paper figures.')
else:
    settings = dict(SETTINGS)
    repetitions = REPEATS if RUN_MODE == 'full' else 1
    if RUN_MODE == 'smoke': settings['n_final'] = 40
    total = len(VARIANTS)*len(METHODS_TO_RUN)*repetitions
    done = 0
    for rule, mode in VARIANTS:
        for repeat in range(repetitions):
            model_args = SimpleNamespace(field_seed=123+1000*repeat,
                                         n_outputs=20, modes_per_dim=5, corr_decay=0.1)
            model, metadata = build_model(model_args)
            for method in METHODS_TO_RUN:
                path = OUTPUT/RUN_MODE/f'{rule}_{mode}_{method}_r{repeat:02d}.npz'
                if path.exists():
                    result = load_result(path)
                    if result.get('configuration') != settings:
                        raise ValueError('Saved settings differ; choose a new OUTPUT folder.')
                else:
                    result = run_2d(model, BOUNDS, method=method, degree_mode=mode,
                                    degree_rule=rule, repeat=repeat, **settings)
                    result['model_metadata'] = metadata
                    result['configuration'] = settings
                    save_result(path, result)
                done += 1
                print(f"[{done}/{total}] {rule}/{mode}, repeat={repeat+1}, {method}: "
                      f"N={result['hist']['n'][-1]}, NRMSE={result['hist']['nrmse'][-1]:.4g}", flush=True)

Reference results are in Data/. Open Plotting.ipynb to reproduce the paper figures.


## 4. Saved paper results

LHS uses a complete independent design at every budget after the shared initial ED. The plotted N is each design size, not the cumulative cost of all LHS designs. New runs are saved separately from Data/.

In [5]:
with np.load(HERE/'Data/results.npz', allow_pickle=False) as reference:
    print('Saved paper budgets:', reference['n'][0], 'to', reference['n'][-1])
    print('Statistics: 50 repetitions; plot means and standard deviations of log10 errors.')

Saved paper budgets: 30 to 1000
Statistics: 50 repetitions; plot means and standard deviations of log10 errors.
